# Exoplanet Tabular Modeling — KOI, TESS, and a Unified Dataset

This notebook demonstrates how to train baseline machine learning models on **exoplanet candidate data** from two major NASA missions — **Kepler (KOI)** and **TESS** — and how to combine them into a unified dataset for improved generalization.

---

### What This Notebook Does  
- Loads tabular features for Kepler (KOI) and TESS candidates using the same consistent schema.  
- Trains baseline models on each dataset separately to understand their individual performance.  
- **New:** Identifies overlapping features between KOI and TESS, standardizes column names, harmonizes labels to `CONFIRMED` / `CANDIDATE`, and unions the datasets.  
- Trains the same baseline models on the **combined dataset** to compare results and observe potential performance gains from data integration.

---

### Why This Matters  
Combining KOI and TESS expands the available training data, helping models learn more robust patterns across different missions. By focusing on **comparable features** (such as orbital period, transit duration, depth, planet radius, equilibrium temperature, and stellar parameters), we maintain scientific consistency while improving generalization.

---

### How To Use  
1. Place both CSVs in the `data/` folder:  
   - `Kepler Object of Interest.csv`  
   - `TESS Project Candidates.csv`  
2. Run cells top to bottom.  
3. If a feature is missing in one dataset, the code automatically drops rows with missing values to keep training simple and reproducible.

---


In [1]:
import numpy as np
import pandas as pd
from collections import OrderedDict

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

# xgboost (optional)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# lightgbm (meta)
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

RANDOM_STATE = 42

def safe_predict_proba(model, X):
    """
    Return class-1 probabilities for binary classifiers.
    Falls back to decision_function if needed.
    """
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        # if returns shape (n,2) take column 1; if (n,) already probs
        return p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.ravel()
    elif hasattr(model, "decision_function"):
        from sklearn.preprocessing import MinMaxScaler
        s = model.decision_function(X).reshape(-1, 1)
        # MinMax to [0,1] as a simple monotonic squashing
        return MinMaxScaler().fit_transform(s).ravel()
    else:
        # last resort: predict labels then make {0,1}
        return model.predict(X).astype(float).ravel()


In [2]:
# function for splitting dataset into train/val/test with index alignment
from sklearn.model_selection import train_test_split

def stratified_split_70_15_15_with_ids(X, y, ids, prefix="", random_state=42):
    """
    Performs a stratified 70/15/15 split with index alignment.
    Adds a prefix (e.g. 'koi', 'tess', 't2') to variable names for clarity.
    """
    stratify_opt = y if len(np.unique(y)) > 1 else None

    X_tr, X_tmp, y_tr, y_tmp, ids_tr, ids_tmp = train_test_split(
        X, y, ids, test_size=0.30, stratify=stratify_opt, random_state=random_state
    )

    stratify_tmp = y_tmp if len(np.unique(y_tmp)) > 1 else None
    X_va, X_te, y_va, y_te, ids_va, ids_te = train_test_split(
        X_tmp, y_tmp, ids_tmp, test_size=0.50, stratify=stratify_tmp, random_state=random_state
    )

    # Sanity checks
    assert X_tr.index.equals(ids_tr.index) and y_tr.index.equals(ids_tr.index)
    assert X_va.index.equals(ids_va.index) and y_va.index.equals(ids_va.index)
    assert X_te.index.equals(ids_te.index) and y_te.index.equals(ids_te.index)

    # Reset indices
    for obj in [X_tr, X_va, X_te, y_tr, y_va, y_te, ids_tr, ids_va, ids_te]:
        obj.reset_index(drop=True, inplace=True)

    print(f"[{prefix.upper()}] Split → train: {len(X_tr)} | val: {len(X_va)} | test: {len(X_te)}")

    # Return as a dictionary with prefix-based keys for neat unpacking
    return {
        f"X_{prefix}_train": X_tr,
        f"X_{prefix}_val": X_va,
        f"X_{prefix}_test": X_te,
        f"y_{prefix}_train": y_tr,
        f"y_{prefix}_val": y_va,
        f"y_{prefix}_test": y_te,
        f"ids_{prefix}_train": ids_tr,
        f"ids_{prefix}_val": ids_va,
        f"ids_{prefix}_test": ids_te
    }


In [3]:
def make_baseline_models(random_state: int = RANDOM_STATE):
    models = {}

    models["rf"] = RandomForestClassifier(
        n_estimators=500, max_depth=None, n_jobs=-1,
        class_weight="balanced_subsample", random_state=random_state
    )

    models["ada"] = AdaBoostClassifier(
        n_estimators=300, learning_rate=0.5, random_state=random_state
    )

    models["gbrt"] = GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=3,
        random_state=random_state
    )

    if HAS_XGB:
        models["xgb"] = XGBClassifier(
            n_estimators=600, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=random_state, n_jobs=-1,
            eval_metric="logloss", objective="binary:logistic",
            tree_method="hist"
        )
    else:
        print("⚠️  XGBoost not available — skipping 'xgb' baseline.")

    return models


In [4]:
# 3 - 
def fit_baselines_and_collect_val_proba(models_dict, X_train, y_train, X_val, y_val, calibrate=False):
    trained = OrderedDict()
    Z_cols, Z_val_parts = [], []

    for name, model in models_dict.items():
        if model is None: 
            print(f"Skipping '{name}' because model is None.")
            continue

        mdl = model
        if calibrate:
            # isotonic can help if baselines are poorly calibrated
            mdl = CalibratedClassifierCV(model, method="isotonic", cv=3)

        mdl.fit(X_train, y_train)
        trained[name] = mdl

        p_val = safe_predict_proba(mdl, X_val).reshape(-1, 1)
        Z_val_parts.append(p_val)
        Z_cols.append(f"{name}_p1")

        auc = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) == 2 else np.nan
        print(f"✓ Trained baseline: {name:>5} | val AUC={auc:.3f}")

    Z_val = np.hstack(Z_val_parts) if Z_val_parts else np.zeros((len(X_val), 0))
    return trained, Z_val, Z_cols


In [5]:
# 4 - meta model
if HAS_LGBM:
    META_MODEL = LGBMClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=-1,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1
    )
else:
    # fallback to logistic regression if LightGBM unavailable
    from sklearn.linear_model import LogisticRegression
    META_MODEL = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)

def train_stack_for_dataset(prefix, X, y, ids, random_state=RANDOM_STATE, calibrate=False):
    """
    End-to-end:
      1) split (70/15/15),
      2) train baselines, collect Z_val,
      3) build Z_test,
      4) train meta on Z_val,
      5) evaluate on Z_test.
    Returns dict with all artifacts.
    """
    # 1) split
    splits = stratified_split_70_15_15_with_ids(X, y, ids, prefix=prefix, random_state=random_state)
    X_train = splits[f"X_{prefix}_train"]; X_val = splits[f"X_{prefix}_val"]; X_test = splits[f"X_{prefix}_test"]
    y_train = splits[f"y_{prefix}_train"]; y_val = splits[f"y_{prefix}_val"]; y_test = splits[f"y_{prefix}_test"]

    # 2) baselines
    BASELINE_MODELS = make_baseline_models(random_state)
    trained_baselines, Z_val, Z_cols = fit_baselines_and_collect_val_proba(
        BASELINE_MODELS, X_train, y_train, X_val, y_val, calibrate=calibrate
    )
    print(f"\n[{prefix.upper()}] Stacking features (VAL) → shape: {Z_val.shape} | columns: {Z_cols}")

    # Optional: baseline metrics on TEST
    print("\n=== Baseline Test Metrics (context) ===")
    for name, mdl in trained_baselines.items():
        p = safe_predict_proba(mdl, X_test)
        yhat = (p >= 0.5).astype(int)
        auc  = roc_auc_score(y_test, p) if len(np.unique(y_test)) == 2 else np.nan
        acc  = accuracy_score(y_test, yhat)
        prec = precision_score(y_test, yhat, zero_division=0)
        rec  = recall_score(y_test, yhat, zero_division=0)
        f1   = f1_score(y_test, yhat, zero_division=0)
        print(f"{name:>5} → AUC={auc:.3f}  ACC={acc:.3f}  PREC={prec:.3f}  REC={rec:.3f}  F1={f1:.3f}")

    # 3) build Z_test from trained baselines
    Z_test_parts = []
    for name, mdl in trained_baselines.items():
        p_test = safe_predict_proba(mdl, X_test).reshape(-1, 1)
        Z_test_parts.append(p_test)
    Z_test = np.hstack(Z_test_parts)
    print(f"\n[{prefix.upper()}] Stacking TEST features shape: {Z_test.shape}")

    # 4) train meta on Z_val
    meta_model = META_MODEL
    meta_model.fit(Z_val, y_val)

    # 5) evaluate meta on Z_test
    p_meta = safe_predict_proba(meta_model, Z_test)
    y_pred = (p_meta >= 0.5).astype(int)

    auc  = roc_auc_score(y_test, p_meta)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    print("\n=== Meta LightGBM (Stacking) Performance ===")
    print(f"AUC  : {auc:.3f}")
    print(f"ACC  : {acc:.3f}")
    print(f"PREC : {prec:.3f}")
    print(f"REC  : {rec:.3f}")
    print(f"F1   : {f1:.3f}")

    return {
        "prefix": prefix,
        "splits": splits,
        "baselines": trained_baselines,
        "Z_val": Z_val,
        "Z_test": Z_test,
        "Z_cols": Z_cols,
        "meta_model": meta_model,
        "meta_metrics": {"auc": auc, "acc": acc, "prec": prec, "rec": rec, "f1": f1}
    }


### Kepler Objects of Interest

In [6]:
import os
# === Config (Tabular KOI features) ============================================
data_path   = "data/Kepler Object of Interest.csv"  # <-- change to your file
target_col  = "koi_disposition"                      # CONFIRMED / CANDIDATE / FALSE POSITIVE

# Core transit/orbital + host-star features
feature_cols_base = [
    "koi_period","koi_time0bk","koi_duration","koi_depth",
    "koi_prad","koi_teq","koi_steff","koi_slogg","koi_srad","koi_kepmag"
]

# HIGH-LEVERAGE ADDITIONS: KOI false-positive flags (binary)
fp_flag_cols = ["koi_fpflag_nt","koi_fpflag_ss","koi_fpflag_co","koi_fpflag_ec"]

# Final tabular feature set for this run
feature_cols = feature_cols_base + fp_flag_cols

# Keep IDs for traceability (never used as features)
f_ids_koi = ["kepid", "kepler_name"]

# Label policy:
#   "confirmed_vs_candidate" → binary (CONFIRMED=1, CANDIDATE=0), drops FALSE POSITIVE
#   "three_class"            → keep all three classes (multiclass)
label_policy = "confirmed_vs_candidate"
positive_label = "CONFIRMED"
negative_label = "CANDIDATE"

# === Load (Excel or CSV) ======================================================
ext = os.path.splitext(data_path)[1].lower()
if ext in (".xls", ".xlsx"):
    df_koi_raw = pd.read_excel(data_path)
else:
    df_koi_raw = pd.read_csv(data_path)

# Normalize column names (strip spaces, keep case)
df_koi_raw.columns = df_koi_raw.columns.astype(str).str.strip()

# Validate required columns exist
required = set([target_col] + feature_cols + f_ids_koi)
missing = sorted([c for c in required if c not in df_koi_raw.columns])
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Select columns: IDs + target + features
df_koi = df_koi_raw[f_ids_koi + [target_col] + feature_cols].copy()

# Optional: enforce label policy
if label_policy == "confirmed_vs_candidate":
    # Keep only CONFIRMED / CANDIDATE; drop others (e.g., FALSE POSITIVE/UNKNOWN)
    keep = {positive_label, negative_label}
    before_lp = len(df_koi)
    df_koi = df_koi[df_koi[target_col].isin(keep)].reset_index(drop=True)
    dropped_lp = before_lp - len(df_koi)

    # Map to binary 1/0 for training
    label_map = {positive_label: 1, negative_label: 0}
    y = df_koi[target_col].map(label_map).astype(int)
else:
    # Multiclass label as-is
    y = df_koi[target_col].astype(str)

# Coerce numeric dtypes for all feature columns (safe)
for c in feature_cols:
    df_koi[c] = pd.to_numeric(df_koi[c], errors="coerce")

# Drop rows with NA in target or any feature
before = len(df_koi)
df_koi = df_koi.dropna(subset=feature_cols).reset_index(drop=True)
y_koi = y.loc[df_koi.index]  # align y with filtered df_koi
dropped_na = before - len(df_koi)

# Quick sanity info
display(df_koi.head())
print(f"Rows kept: {len(df_koi)}  |  Dropped (label policy): {dropped_lp if label_policy=='confirmed_vs_candidate' else 0}  |  Dropped due to NA: {dropped_na}")
print(f"Features used ({len(feature_cols)}): {feature_cols}")
print(f"ID columns retained (not used as features): {f_ids_koi}")

# Split-ready matrices
X_koi = df_koi[feature_cols].copy()  # ✅ Features
ids_koi_data = df_koi[f_ids_koi].copy()  # ✅ IDs


,kepid,kepler_name,koi_disposition,koi_period,koi_time0bk,koi_duration,koi_depth,koi_prad,koi_teq,koi_steff,koi_slogg,koi_srad,koi_kepmag,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec
0,10797460,Kepler-227 b,CONFIRMED,9.488036,170.538750,2.9575,616.0,2.26,793.0,5455.0,4.467,0.927,15.347,0,0,0,0
1,10797460,Kepler-227 c,CONFIRMED,54.418383,162.513840,4.5070,875.0,2.83,443.0,5455.0,4.467,0.927,15.347,0,0,0,0
2,10811496,NaN,CANDIDATE,19.899140,175.850252,1.7822,10800.0,14.60,638.0,5853.0,4.544,0.868,15.436,0,0,0,0
3,10854555,Kepler-664 b,CONFIRMED,2.525592,171.595550,1.6545,603.0,2.75,1406.0,6031.0,4.438,1.046,15.509,0,0,0,0
4,10872983,Kepler-228 d,CONFIRMED,11.094321,171.201160,4.5945,1520.0,3.90,835.0,6046.0,4.486,0.972,15.714,0,0,0,0


Rows kept: 4619  |  Dropped (label policy): 4839  |  Dropped due to NA: 106
Features used (14): ['koi_period', 'koi_time0bk', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']
ID columns retained (not used as features): ['kepid', 'kepler_name']


In [7]:
# Example (koi) — assuming you already built X_tess, y_tess, ids_tess
art_koi = train_stack_for_dataset(prefix="koi", X=X_koi, y=y_koi, ids=ids_koi_data)

[KOI] Split → train: 3233 | val: 693 | test: 693
✓ Trained baseline:    rf | val AUC=0.774
✓ Trained baseline:   ada | val AUC=0.771
✓ Trained baseline:  gbrt | val AUC=0.791
✓ Trained baseline:   xgb | val AUC=0.753

[KOI] Stacking features (VAL) → shape: (693, 4) | columns: ['rf_p1', 'ada_p1', 'gbrt_p1', 'xgb_p1']

=== Baseline Test Metrics (context) ===
   rf → AUC=0.756  ACC=0.700  PREC=0.713  REC=0.830  F1=0.767
  ada → AUC=0.728  ACC=0.680  PREC=0.679  REC=0.876  F1=0.765
 gbrt → AUC=0.759  ACC=0.707  PREC=0.722  REC=0.825  F1=0.770
  xgb → AUC=0.737  ACC=0.697  PREC=0.724  REC=0.791  F1=0.756

[KOI] Stacking TEST features shape: (693, 4)
[LightGBM] [Info] Number of positive: 412, number of negative: 281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000180 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 926
[LightGBM] [Info] Number of data points in the train set: 693, number of used feature

c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [55]:
# --- assemble per-model predictions table for the TEST split -------------------
import numpy as np
import pandas as pd

# display names for columns
MODEL_NAME_MAP = {
    "gbrt": "GradientBoosting",
    "rf": "RandomForest",
    "ada": "AdaBoost",
    "xgb": "XGBoost",          # may be absent if HAS_XGB=False
}
META_NAME = "LightGBM"         # your META_MODEL

def _to_label(p, thr=0.5):
    return np.where(p >= thr, "CONFIRMED", "CANDIDATE")

def build_model_outputs_table(art, df_koi_raw, prefix="koi"):
    """Create a table like:
       kepid | kepoi_name | kepler_name | GradientBoosting | RandomForest | XGBoost | LightGBM | Actual | koi_score
    """
    splits = art["splits"]
    X_test = splits[f"X_{prefix}_test"]
    y_test = splits[f"y_{prefix}_test"]
    # ids for test rows (prefer split-provided ids; else align from original ids df)
    ids_key = f"ids_{prefix}_test"
    if ids_key in splits:
        ids_test = splits[ids_key].copy()
    else:
        # fallback: use whatever IDs you passed (assuming available globally as ids_koi_data)
        ids_test = ids_koi_data.loc[X_test.index].copy()

    # ensure we have kepoi_name: try from ids, else fetch from raw if available
    if "kepoi_name" not in ids_test.columns:
        if "kepoi_name" in df_koi_raw.columns:
            ids_test = ids_test.copy()
            ids_test["kepoi_name"] = df_koi_raw.loc[X_test.index, "kepoi_name"].values
        else:
            ids_test = ids_test.copy()
            ids_test["kepoi_name"] = np.nan

    out = pd.DataFrame({
        "Kepid": ids_test.get("kepid", pd.Series(index=X_test.index, dtype=object)),
        "KepoiName": ids_test.get("kepoi_name", pd.Series(index=X_test.index, dtype=object)),
        "KeplerName": ids_test.get("kepler_name", pd.Series(index=X_test.index, dtype=object)),
    }, index=X_test.index)

    # collect probabilities from trained baselines
    trained = art["baselines"]
    for key, mdl in trained.items():
        disp = MODEL_NAME_MAP.get(key, key)
        p = safe_predict_proba(mdl, X_test)
        out[disp] = _to_label(p)

    # meta predictions from stacking
    # art already built Z_test in the same order as X_test
    Z_test = art["Z_test"]
    p_meta = safe_predict_proba(art["meta_model"], Z_test)
    out[META_NAME] = _to_label(p_meta)
    out["Actual"] = np.where(y_test.values.astype(int) == 1, "CONFIRMED", "CANDIDATE")
    out["koi_score"] = p_meta.round(3)  # probability of CONFIRMED

    # reorder columns per your example (include XGBoost if present)
    cols = ["Kepid", "KepoiName", "KeplerName",
            "GradientBoosting", "RandomForest"]
    if "XGBoost" in out.columns:
        cols.append("XGBoost")
    cols += [META_NAME, "Actual", "koi_score"]

    out = out[cols].reset_index(drop=True)
    return out

# build the table
koi_model_outputs = build_model_outputs_table(art_koi, df_koi_raw, prefix="koi")

# quick peek
print(koi_model_outputs.head(10).to_string(index=False))

# optional: save to CSV for your blog/dashboard
koi_model_outputs.to_csv("koi_model_outputs_test.csv", index=False)


  Kepid KepoiName    KeplerName GradientBoosting RandomForest   XGBoost  LightGBM    Actual  koi_score
3764879 K00752.01           NaN        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.999
8456679 K00752.02 Kepler-1990 c        CONFIRMED    CANDIDATE CANDIDATE CONFIRMED CONFIRMED      0.979
8612847 K00753.01  Kepler-800 b        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.986
7595157 K00754.01  Kepler-605 b        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.999
8950853 K00755.01  Kepler-839 b        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.992
9150827 K00756.01 Kepler-1749 b        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.906
6922244 K00756.02    Kepler-8 b        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.977
7466270 K00756.03           NaN        CONFIRMED    CONFIRMED CONFIRMED CONFIRMED CONFIRMED      0.927
9782691 K00114.01  Kepler-193 b        CONFIRMED    CONFIRMED CONFIRMED C

c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### TESS Project Candidates

In [8]:
# === Config (Tabular TESS features) ==========================================
data_path   = "data/TESS Project Candidates.csv"   # <-- change to your file
target_col  = "tfopwg_disp"                        # CP / PC

# Core transit/orbital + host-star features
feature_cols_base = [
    "pl_orbper","pl_trandurh","pl_trandep",
    "pl_rade","pl_eqt","pl_insol",
    "st_teff","st_logg","st_rad","st_tmag","st_dist"
]

# HIGH-LEVERAGE ADDITIONS: (no TESS binary FP flags provided in this table)
fp_flag_cols = []  # kept for API-compat with KOI script

# Final tabular feature set for this run
feature_cols = feature_cols_base + fp_flag_cols

# Keep IDs for traceability (never used as features)
f_ids_tess = ["toi", "tid"]

# Label policy:
#   "confirmed_vs_candidate" → binary (CP=1, PC=0)
label_policy = "confirmed_vs_candidate"
positive_label = "CP"
negative_label = "PC"

# === Load (Excel or CSV) ======================================================
ext = os.path.splitext(data_path)[1].lower()
if ext in (".xls", ".xlsx"):
    df_tess_raw = pd.read_excel(data_path)
else:
    df_tess_raw = pd.read_csv(data_path)

# Normalize column names (strip spaces, keep case)
df_tess_raw.columns = df_tess_raw.columns.astype(str).str.strip()

# Validate required columns exist
required = set([target_col] + feature_cols + f_ids_tess)
missing = sorted([c for c in required if c not in df_tess_raw.columns])
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Select columns: IDs + target + features
df_tess = df_tess_raw[f_ids_tess + [target_col] + feature_cols].copy()

# Optional: enforce label policy
if label_policy == "confirmed_vs_candidate":
    # Keep only CONFIRMED / CANDIDATE; drop others (e.g., FALSE POSITIVE/UNKNOWN)
    keep = {positive_label, negative_label}
    before_lp = len(df_tess)
    df_tess = df_tess[df_tess[target_col].isin(keep)].reset_index(drop=True)
    dropped_lp = before_lp - len(df_tess)

    # Map to binary 1/0 for training
    label_map = {positive_label: 1, negative_label: 0}
    y = df_tess[target_col].map(label_map).astype(int)
else:
    # Multiclass label as-is
    y = df_tess[target_col].astype(str)

# Coerce numeric dtypes for all feature columns (safe)
for c in feature_cols:
    df_tess[c] = pd.to_numeric(df_tess[c], errors="coerce")

# Drop rows with NA in target or any feature
before = len(df_tess)
df_tess = df_tess.dropna(subset=feature_cols).reset_index(drop=True)
y_tess = y.loc[df_tess.index]  # align y with filtered df_tess
dropped_na = before - len(df_tess)

# Quick sanity info
display(df_tess.head())
print(f"Rows kept: {len(df_tess)}  |  Dropped (label policy): {dropped_lp if label_policy=='confirmed_vs_candidate' else 0}  |  Dropped due to NA: {dropped_na}")
print(f"Features used ({len(feature_cols)}): {feature_cols}")
print(f"ID columns retained (not used as features): {f_ids_tess}")

# Split-ready matrices
X_tess = df_tess[feature_cols].copy()  
ids_tess_data = df_tess[f_ids_tess].copy()  


,toi,tid,tfopwg_disp,pl_orbper,pl_trandurh,pl_trandep,pl_rade,pl_eqt,pl_insol,st_teff,st_logg,st_rad,st_tmag,st_dist
0,1001.01,88863718,PC,1.931646,3.166000,1286.00000,11.215400,4045.000000,44464.500000,7070.0,4.03,2.01000,9.42344,295.8620
1,1007.01,65212867,PC,6.998921,3.953000,2840.00000,14.775200,1282.000000,448.744000,6596.0,3.71,2.70000,8.87759,283.2910
2,1011.01,114018671,PC,2.470498,2.191000,250.00000,1.446560,1364.000000,575.597000,5413.7,4.46,0.94000,8.23880,52.6200
3,1019.01,341420329,PC,5.234101,3.707814,20796.94163,24.293141,1574.838185,1453.666385,7645.0,4.30,1.56203,10.63480,399.5440
4,1027.01,20318757,PC,3.283456,1.537000,1558.00000,2.876480,992.000000,161.092000,4272.0,4.60,0.68000,10.21030,56.3449


Rows kept: 4669  |  Dropped (label policy): 2341  |  Dropped due to NA: 693
Features used (11): ['pl_orbper', 'pl_trandurh', 'pl_trandep', 'pl_rade', 'pl_eqt', 'pl_insol', 'st_teff', 'st_logg', 'st_rad', 'st_tmag', 'st_dist']
ID columns retained (not used as features): ['toi', 'tid']


In [9]:
# train tess
art_tess = train_stack_for_dataset(prefix="tess", X=X_tess, y=y_tess, ids=ids_tess_data)

[TESS] Split → train: 3268 | val: 700 | test: 701
✓ Trained baseline:    rf | val AUC=0.668
✓ Trained baseline:   ada | val AUC=0.604
✓ Trained baseline:  gbrt | val AUC=0.622
✓ Trained baseline:   xgb | val AUC=0.622

[TESS] Stacking features (VAL) → shape: (700, 4) | columns: ['rf_p1', 'ada_p1', 'gbrt_p1', 'xgb_p1']

=== Baseline Test Metrics (context) ===
   rf → AUC=0.623  ACC=0.863  PREC=0.000  REC=0.000  F1=0.000
  ada → AUC=0.621  ACC=0.866  PREC=0.000  REC=0.000  F1=0.000
 gbrt → AUC=0.612  ACC=0.864  PREC=0.400  REC=0.021  F1=0.040
  xgb → AUC=0.592  ACC=0.849  PREC=0.125  REC=0.021  F1=0.036

[TESS] Stacking TEST features shape: (701, 4)
[LightGBM] [Info] Number of positive: 93, number of negative: 607
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 624
[LightGBM] [Info] Number of data points in the train set: 700, number of used featu

c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
# --- assemble per-model predictions table for the TESS TEST split -------------
import numpy as np
import pandas as pd

# consistent display names (same as KOI)
MODEL_NAME_MAP = {
    "gbrt": "GradientBoosting",
    "rf": "RandomForest",
    "ada": "AdaBoost",
    "xgb": "XGBoost",      # present only if you included XGB in baselines
}
META_NAME = "LightGBM"     # your META_MODEL display name

def _to_label(p, thr=0.5):
    # probability → label names for readability
    return np.where(p >= thr, "CONFIRMED", "CANDIDATE")

def build_model_outputs_table_tess(art, df_tess_raw, prefix="tess"):
    """
    Returns a DataFrame:
    TOI | TID | GradientBoosting | RandomForest | [XGBoost] | LightGBM | Actual | tess_score
    """
    splits = art["splits"]
    X_test = splits[f"X_{prefix}_test"]
    y_test = splits[f"y_{prefix}_test"]  # 1=CONFIRMED, 0=CANDIDATE

    # ids for test rows (prefer split-provided; else align from ids_tess_data)
    ids_key = f"ids_{prefix}_test"
    if ids_key in splits:
        ids_test = splits[ids_key].copy()
    else:
        # fallback if your splitter didn't store ids_* in splits
        ids_test = ids_tess_data.loc[X_test.index].copy()

    # minimal ID surface: ensure TOI and TID columns exist (case-insensitive safety)
    col_map = {c.lower(): c for c in ids_test.columns}
    toi_col = col_map.get("toi", "toi")
    tid_col = col_map.get("tid", "tid")
    if toi_col not in ids_test.columns and "toi" in df_tess_raw.columns:
        ids_test[toi_col] = df_tess_raw.loc[X_test.index, "toi"].values
    if tid_col not in ids_test.columns and "tid" in df_tess_raw.columns:
        ids_test[tid_col] = df_tess_raw.loc[X_test.index, "tid"].values

    out = pd.DataFrame({
        "TOI": ids_test.get(toi_col, pd.Series(index=X_test.index, dtype=object)),
        "TID": ids_test.get(tid_col, pd.Series(index=X_test.index, dtype=object)),
    }, index=X_test.index)

    # per-baseline predictions (label at 0.5 threshold)
    trained = art["baselines"]
    for key, mdl in trained.items():
        disp = MODEL_NAME_MAP.get(key, key)
        p = safe_predict_proba(mdl, X_test)           # probability of CONFIRMED (class=1)
        out[disp] = _to_label(p)

    # meta (stacking) predictions
    Z_test = art["Z_test"]                            # built in train_stack_for_dataset
    p_meta = safe_predict_proba(art["meta_model"], Z_test)
    out[META_NAME] = _to_label(p_meta)
    out["Actual"] = np.where(y_test.values.astype(int) == 1, "CONFIRMED", "CANDIDATE")
    out["tess_score"] = p_meta.round(3)               # probability of CONFIRMED

    # match column order used in your KOI table
    cols = ["TOI", "TID", "GradientBoosting", "RandomForest"]
    if "XGBoost" in out.columns:
        cols.append("XGBoost")
    cols += [META_NAME, "Actual", "tess_score"]

    out = out[cols].reset_index(drop=True)
    return out

# build the table
tess_model_outputs = build_model_outputs_table_tess(art_tess, df_tess_raw, prefix="tess")

# quick peek
print(tess_model_outputs.head(10).to_string(index=False))

# optional: save
tess_model_outputs.to_csv("tess_model_outputs_test.csv", index=False)



    TOI       TID GradientBoosting RandomForest   XGBoost  LightGBM    Actual  tess_score
3660.01  67517533        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.089
3644.01 445496795        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.014
6114.01 419840740        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.000
 732.01  36724087        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.002
7391.01 258776466        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.091
4129.01 123357034        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.002
3878.01 141275395        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.004
1437.01 198356533        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.089
2875.01 385658469        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.001
5350.01  68808155        CANDIDATE    CANDIDATE CANDIDATE CANDIDATE CANDIDATE       0.083


c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
# minimal imports used by the combined train step
import os
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

In [12]:
# define config for koi (kepler) and tess
koi_path  = Path("data/Kepler Object of Interest.csv")
tess_path = Path("data/TESS Project Candidates.csv")

# koi columns
koi_target_col = "koi_disposition"  # values include CONFIRMED / CANDIDATE / FALSE POSITIVE
koi_keep_map = {
    "koi_period":   "period",
    "koi_duration": "duration_hours",
    "koi_depth":    "depth_ppm",
    "koi_prad":     "prad_re",
    "koi_teq":      "teq_k",
    "koi_steff":    "st_teff",
    "koi_slogg":    "st_logg",
    "koi_srad":     "st_rad"
    # note: koi_time0bk and koi_kepmag are not directly comparable to tess fields; we skip for union
}

# tess columns
tess_target_col = "tfopwg_disp"  # common encodings: CP/PC or CF/CN
tess_keep_map = {
    "pl_orbper":   "period",
    "pl_trandurh": "duration_hours",
    "pl_trandep":  "depth_ppm",
    "pl_rade":     "prad_re",
    "pl_eqt":      "teq_k",
    "st_teff":     "st_teff",
    "st_logg":     "st_logg",
    "st_rad":      "st_rad"
    # note: st_tmag, st_dist, pl_insol have no koi match; we skip for union
}

# label normalization function
def normalize_label(val):
    # map all known variants to confirmed/candidate
    mapping = {
        "CONFIRMED": "CONFIRMED",
        "CANDIDATE": "CANDIDATE",
        "CP": "CONFIRMED", "CF": "CONFIRMED",
        "PC": "CANDIDATE", "CN": "CANDIDATE"
    }
    if pd.isna(val):
        return None
    # normalize to upper for safety
    up = str(val).strip().upper()
    return mapping.get(up, None)  # return None for anything else (e.g., FALSE POSITIVE)

In [13]:
# load koi
koi_df = pd.read_csv(koi_path)
# keep only columns we need + target
koi_cols = list(koi_keep_map.keys()) + [koi_target_col]
koi_df = koi_df[koi_cols].copy()
koi_df.rename(columns=koi_keep_map, inplace=True)
koi_df["disposition"] = koi_df[koi_target_col].apply(normalize_label)
koi_df = koi_df.drop(columns=[koi_target_col])
# drop rows not labeled as confirmed/candidate
koi_df = koi_df[koi_df["disposition"].isin(["CONFIRMED", "CANDIDATE"])]

# load tess
tess_df = pd.read_csv(tess_path)
# keep only columns we need + target
tess_cols = list(tess_keep_map.keys()) + [tess_target_col]
tess_df = tess_df[tess_cols].copy()
tess_df.rename(columns=tess_keep_map, inplace=True)
tess_df["disposition"] = tess_df[tess_target_col].apply(normalize_label)
tess_df = tess_df.drop(columns=[tess_target_col])
# drop rows not labeled as confirmed/candidate
tess_df = tess_df[tess_df["disposition"].isin(["CONFIRMED", "CANDIDATE"])]

# identify the intersection of shared columns (features + the unified label)
shared_feature_cols = sorted(set(koi_df.columns).intersection(set(tess_df.columns)) - {"disposition"})
print("shared features:", shared_feature_cols)

# reduce to shared schema and concatenate
koi_small  = koi_df[shared_feature_cols + ["disposition"]].copy()
tess_small = tess_df[shared_feature_cols + ["disposition"]].copy()
combo_df = pd.concat([koi_small.assign(source="KOI"), tess_small.assign(source="TESS")], axis=0, ignore_index=True)

# drop rows with any missing values across selected features to keep training simple
before = len(combo_df)
combo_df = combo_df.dropna(subset=shared_feature_cols + ["disposition"])
after = len(combo_df)
print(f"dropped {before - after} rows with missing values (kept {after})")

# quick sanity check
print(combo_df["disposition"].value_counts())
combo_df.head(3)

# === ensure ID columns exist in cleaned tables (PLACE A) ===
for c in ["kepid", "kepoi_name", "kepler_name"]:
    if c not in df_koi.columns:
        df_koi[c] = np.nan

for c in ["toi", "tid"]:
    if c not in df_tess.columns:
        df_tess[c] = np.nan

# Optional recovery if kepoi_name is missing and you still have the raw KOI CSV:
# raw_koi = pd.read_csv(koi_path, low_memory=False, usecols=["kepid","kepoi_name","kepler_name"])
# df_koi = df_koi.merge(raw_koi.drop_duplicates("kepid"), on="kepid", how="left", suffixes=("","_raw"))
# for c in ["kepoi_name","kepler_name"]:
#     if f"{c}_raw" in df_koi.columns:
#         df_koi[c] = df_koi[c].fillna(df_koi[f"{c}_raw"])
#         df_koi.drop(columns=[f"{c}_raw"], inplace=True)


shared features: ['depth_ppm', 'duration_hours', 'period', 'prad_re', 'st_logg', 'st_rad', 'st_teff', 'teq_k']
dropped 779 rows with missing values (kept 9308)
disposition
CANDIDATE    5896
CONFIRMED    3412
Name: count, dtype: int64


In [14]:
# === build ID map in KOI→TESS order, align to X, and split X/y/ids together (PLACE B) ===
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def stratified_split_with_ids(X, y, ids, prefix="combo", test_size=0.176, val_size=0.176, random_state=42):
    # train+val vs test
    X_trv, X_te, y_trv, y_te, ids_trv, ids_te = train_test_split(
        X, y, ids, test_size=test_size, stratify=y, random_state=random_state
    )
    # train vs val
    val_frac = val_size / (1 - test_size)
    X_tr, X_va, y_tr, y_va, ids_tr, ids_va = train_test_split(
        X_trv, y_trv, ids_trv, test_size=val_frac, stratify=y_trv, random_state=random_state
    )
    return {
        f"X_{prefix}_train": X_tr,  f"y_{prefix}_train": y_tr,  f"ids_{prefix}_train": ids_tr,
        f"X_{prefix}_val":   X_va,  f"y_{prefix}_val":   y_va,  f"ids_{prefix}_val":   ids_va,
        f"X_{prefix}_test":  X_te,  f"y_{prefix}_test":  y_te,  f"ids_{prefix}_test":  ids_te,
    }

# Build KOI/TESS ID frames (use the ID columns you ensured in PLACE A)
koi_ids = pd.DataFrame({
    "source": "KOI",
    "kepid":       df_koi["kepid"],
    "kepoi_name":  df_koi["kepoi_name"],
    "kepler_name": df_koi["kepler_name"],
    "toi":         np.nan,
    "tid":         np.nan,
})
tess_ids = pd.DataFrame({
    "source":      "TESS",
    "kepid":       np.nan,
    "kepoi_name":  np.nan,
    "kepler_name": np.nan,
    "toi":         df_tess["toi"],
    "tid":         df_tess["tid"],
})

# Concatenate in the SAME order you built combo_df (KOI first → TESS)
ids_combo_all = pd.concat([koi_ids, tess_ids], axis=0, ignore_index=True)

# Align by POSITION to X, then give IDs the same index as X
ids_combo_data = ids_combo_all.iloc[:len(X)].copy()
ids_combo_data.index = X.index

# Split X/y/ids together (so the test split has the right IDs)
splits = stratified_split_with_ids(X, y, ids_combo_data, prefix="combo", random_state=42)


NameError: name 'X' is not defined

In [16]:
combo_df.head()

,depth_ppm,duration_hours,period,prad_re,st_logg,st_rad,st_teff,teq_k,disposition,source
0,616.0,2.9575,9.488036,2.26,4.467,0.927,5455.0,793.0,CONFIRMED,KOI
1,875.0,4.5070,54.418383,2.83,4.467,0.927,5455.0,443.0,CONFIRMED,KOI
2,10800.0,1.7822,19.899140,14.60,4.544,0.868,5853.0,638.0,CANDIDATE,KOI
3,603.0,1.6545,2.525592,2.75,4.438,1.046,6031.0,1406.0,CONFIRMED,KOI
4,1520.0,4.5945,11.094321,3.90,4.486,0.972,6046.0,835.0,CONFIRMED,KOI


In [15]:
# --- unified training using 70/15/15 stratified split with ids ----------------
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support, classification_report
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# optional xgboost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

random_state = 42

# 1) prepare X / y and a simple ids series (fall back to source+row if no explicit ids)
X = combo_df[shared_feature_cols].copy()
y = combo_df["disposition"].map({"CONFIRMED": 1, "CANDIDATE": 0}).astype(int)

if "source" in combo_df.columns:
    ids_series = combo_df.index.to_series().map(lambda i: f"{combo_df.loc[i, 'source']}_{i}")
else:
    ids_series = pd.Series(np.arange(len(combo_df)), index=combo_df.index, name="row_id")

# 2) split with your helper (keeps indices aligned)
splits = stratified_split_70_15_15_with_ids(X, y, ids_series, prefix="combo", random_state=random_state)

X_train = splits["X_combo_train"]
X_val   = splits["X_combo_val"]
X_test  = splits["X_combo_test"]
y_train = splits["y_combo_train"]
y_val   = splits["y_combo_val"]
y_test  = splits["y_combo_test"]

# 3) baselines with requested parameters
baselines = {
    "rf": RandomForestClassifier(
        n_estimators=500, max_depth=None, n_jobs=-1,
        class_weight="balanced_subsample", random_state=random_state
    ),
    "gbrt": GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=3,
        random_state=random_state
    ),
    "ada": AdaBoostClassifier(
        n_estimators=300, learning_rate=0.5, random_state=random_state
    ),
}

if HAS_XGB:
    baselines["xgb"] = XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=random_state, n_jobs=-1,
        eval_metric="logloss", objective="binary:logistic",
        tree_method="hist"
    )

# 4) train/evaluate with scaler+model pipeline; report VAL AUC during model selection
val_reports = {}
test_reports = {}

for name, clf in baselines.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", clf)])
    pipe.fit(X_train, y_train)

    # validation metrics (for model selection)
    yv_proba = pipe.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, yv_proba)
    print(f"✓ Trained baseline: {name:>4} | val AUC={val_auc:.3f}")

    # test metrics (context)
    yt_pred  = pipe.predict(X_test)
    yt_proba = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, yt_proba)
    acc = accuracy_score(y_test, yt_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, yt_pred, average="binary", zero_division=0)

    val_reports[name] = {"val_auc": val_auc}
    test_reports[name] = {"auc": test_auc, "acc": acc, "prec": prec, "rec": rec, "f1": f1}

# 5) concise summaries
val_df  = pd.DataFrame(val_reports).T.sort_values("val_auc", ascending=False)
test_df = pd.DataFrame(test_reports).T.sort_values("auc", ascending=False)

print("\n=== Baseline Validation Metrics ===")
print(val_df.round(3))

print("\n=== Baseline Test Metrics (context) ===")
print(test_df.round(3))


[COMBO] Split → train: 6515 | val: 1396 | test: 1397
✓ Trained baseline:   rf | val AUC=0.838
✓ Trained baseline: gbrt | val AUC=0.826
✓ Trained baseline:  ada | val AUC=0.790
✓ Trained baseline:  xgb | val AUC=0.833

=== Baseline Validation Metrics ===
      val_auc
rf      0.838
xgb     0.833
gbrt    0.826
ada     0.790

=== Baseline Test Metrics (context) ===
        auc    acc   prec    rec     f1
xgb   0.839  0.773  0.684  0.707  0.695
rf    0.836  0.770  0.688  0.678  0.683
gbrt  0.827  0.772  0.673  0.732  0.702
ada   0.792  0.739  0.633  0.688  0.659


In [17]:
# --- unified training using 70/15/15 stratified split with ids ----------------
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support, classification_report
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression  # fallback meta

# optional xgboost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# optional lightgbm (preferred meta)
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

random_state = 42
PREFIX = "combo"

# 1) prepare X / y and a simple ids series (fall back to source+row if no explicit ids)
X = combo_df[shared_feature_cols].copy()
y = combo_df["disposition"].map({"CONFIRMED": 1, "CANDIDATE": 0}).astype(int)

if "source" in combo_df.columns:
    ids_series = combo_df.index.to_series().map(lambda i: f"{combo_df.loc[i, 'source']}_{i}")
else:
    ids_series = pd.Series(np.arange(len(combo_df)), index=combo_df.index, name="row_id")

# 2) split with your helper (keeps indices aligned)
splits = stratified_split_70_15_15_with_ids(X, y, ids_series, prefix=PREFIX, random_state=random_state)

X_train = splits[f"X_{PREFIX}_train"]
X_val   = splits[f"X_{PREFIX}_val"]
X_test  = splits[f"X_{PREFIX}_test"]
y_train = splits[f"y_{PREFIX}_train"]
y_val   = splits[f"y_{PREFIX}_val"]
y_test  = splits[f"y_{PREFIX}_test"]

# 3) baselines with requested parameters
baselines = {
    "rf": RandomForestClassifier(
        n_estimators=500, max_depth=None, n_jobs=-1,
        class_weight="balanced_subsample", random_state=random_state
    ),
    "gbrt": GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=3,
        random_state=random_state
    ),
    "ada": AdaBoostClassifier(
        n_estimators=300, learning_rate=0.5, random_state=random_state
    ),
}
if HAS_XGB:
    baselines["xgb"] = XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=random_state, n_jobs=-1,
        eval_metric="logloss", objective="binary:logistic",
        tree_method="hist"
    )

# 4) train/evaluate baselines; keep fitted models for stacking
val_reports = {}
test_reports = {}
fitted_pipes = {}

for name, clf in baselines.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", clf)])
    pipe.fit(X_train, y_train)
    fitted_pipes[name] = pipe

    # validation metrics (selection signal)
    yv_proba = pipe.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, yv_proba)
    print(f"✓ Trained baseline: {name:>4} | val AUC={val_auc:.3f}")

    # test metrics (context)
    yt_pred  = pipe.predict(X_test)
    yt_proba = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, yt_proba)
    acc = accuracy_score(y_test, yt_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, yt_pred, average="binary", zero_division=0)

    val_reports[name] = {"val_auc": val_auc}
    test_reports[name] = {"auc": test_auc, "acc": acc, "prec": prec, "rec": rec, "f1": f1}

# 5) concise baseline summaries
val_df  = pd.DataFrame(val_reports).T.sort_values("val_auc", ascending=False)
test_df = pd.DataFrame(test_reports).T.sort_values("auc", ascending=False)

print("\n=== Baseline Validation Metrics ===")
print(val_df.round(3))

print("\n=== Baseline Test Metrics (context) ===")
print(test_df.round(3))

# 6) build stacking features Z_val / Z_test from fitted baselines
Z_cols = [f"{k}_p1" for k in fitted_pipes.keys()]
Z_val  = np.column_stack([mdl.predict_proba(X_val)[:, 1]  for mdl in fitted_pipes.values()])
Z_test = np.column_stack([mdl.predict_proba(X_test)[:, 1] for mdl in fitted_pipes.values()])
print(f"\n[{PREFIX.upper()}] Stacking features (VAL) → shape: {Z_val.shape} | columns: {Z_cols}")
print(f"[{PREFIX.upper()}] Stacking TEST features shape: {Z_test.shape}")

# 7) train meta model on Z_val, evaluate on Z_test
if HAS_LGBM:
    META_MODEL = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=-1,
        subsample=0.9, colsample_bytree=0.9,
        random_state=random_state
    )
else:
    META_MODEL = LogisticRegression(max_iter=1000, random_state=random_state)

META_MODEL.fit(Z_val, y_val)

p_meta = META_MODEL.predict_proba(Z_test)[:, 1] if hasattr(META_MODEL, "predict_proba") else META_MODEL.decision_function(Z_test)
y_pred = (p_meta >= 0.5).astype(int)

auc  = roc_auc_score(y_test, p_meta)
acc  = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

print("\n=== Meta LightGBM (Stacking) Performance ===" if HAS_LGBM else "\n=== Meta Logistic Regression (Stacking) Performance ===")
print(f"AUC  : {auc:.3f}")
print(f"ACC  : {acc:.3f}")
print(f"PREC : {prec:.3f}")
print(f"REC  : {rec:.3f}")
print(f"F1   : {f1:.3f}")


[COMBO] Split → train: 6515 | val: 1396 | test: 1397
✓ Trained baseline:   rf | val AUC=0.838
✓ Trained baseline: gbrt | val AUC=0.826
✓ Trained baseline:  ada | val AUC=0.790
✓ Trained baseline:  xgb | val AUC=0.833

=== Baseline Validation Metrics ===
      val_auc
rf      0.838
xgb     0.833
gbrt    0.826
ada     0.790

=== Baseline Test Metrics (context) ===
        auc    acc   prec    rec     f1
xgb   0.839  0.773  0.684  0.707  0.695
rf    0.836  0.770  0.688  0.678  0.683
gbrt  0.827  0.772  0.673  0.732  0.702
ada   0.792  0.739  0.633  0.688  0.659

[COMBO] Stacking features (VAL) → shape: (1396, 4) | columns: ['rf_p1', 'gbrt_p1', 'ada_p1', 'xgb_p1']
[COMBO] Stacking TEST features shape: (1397, 4)
[LightGBM] [Info] Number of positive: 512, number of negative: 884
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000204 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1012
[LightGBM] [Info] Nu

c:\Users\roger\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [43]:
# make sure these exist from your earlier prep
# df_koi, df_tess  → cleaned/filtered tables used to build combo_df
# combo_df         → your unified feature/label table (KOI rows first, then TESS), same order as features

import numpy as np
import pandas as pd

# --- ensure identifier columns exist so they don't collapse to all-NaN ----------
for c in ["kepid", "kepoi_name", "kepler_name"]:
    if c not in df_koi.columns:
        df_koi[c] = np.nan
for c in ["toi", "tid"]:
    if c not in df_tess.columns:
        df_tess[c] = np.nan

# --- ids from KOI side ---------------------------------------------------------
koi_ids = pd.DataFrame({
    "source": "KOI",
    "kepid":       df_koi["kepid"],
    "kepoi_name":  df_koi["kepoi_name"],
    "kepler_name": df_koi["kepler_name"],
    "toi":         np.nan,
    "tid":         np.nan,
})

# --- ids from TESS side --------------------------------------------------------
tess_ids = pd.DataFrame({
    "source":      "TESS",
    "kepid":       np.nan,
    "kepoi_name":  np.nan,
    "kepler_name": np.nan,
    "toi":         df_tess["toi"],
    "tid":         df_tess["tid"],
})

# --- stack in the SAME order as features (KOI first, then TESS) ----------------
# use ignore_index=True so we can align by position to combo_df regardless of original indices
ids_combo_all = pd.concat([koi_ids, tess_ids], axis=0, ignore_index=True)

# --- align to combo_df by POSITION (avoid .loc KeyErrors) ----------------------
n = len(combo_df)

if len(ids_combo_all) < n:
    # pad with NaN rows if IDs are shorter (e.g., you filtered features more than IDs)
    pad = pd.DataFrame(np.nan, index=np.arange(n - len(ids_combo_all)), columns=ids_combo_all.columns)
    # if combo_df has a 'source' column, preserve its KOI/TESS labels for padded rows
    if "source" in combo_df.columns:
        pad["source"] = combo_df["source"].iloc[len(ids_combo_all):].values
    ids_combo_all = pd.concat([ids_combo_all, pad], axis=0, ignore_index=True)
elif len(ids_combo_all) > n:
    # trim extra rows if IDs are longer
    ids_combo_all = ids_combo_all.iloc[:n].copy()

# make indices match exactly
ids_combo_data = ids_combo_all.copy()
ids_combo_data.index = combo_df.index

# quick sanity
print("ids non-null counts:")
print(ids_combo_data[["source","kepid","kepoi_name","kepler_name","toi","tid"]].notna().sum())


ids non-null counts:
source         9308
kepid          4619
kepoi_name        0
kepler_name    2743
toi            4669
tid            4669
dtype: int64


In [44]:
# --- rebuild id map from original KOI + TESS tables, align to combo_df, train, and save ----
import numpy as np
import pandas as pd

# extract identifiers safely
koi_ids = pd.DataFrame({
    "source": "KOI",
    "kepid":       df_koi.get("kepid", np.nan),
    "kepoi_name":  df_koi.get("kepoi_name", np.nan),
    "kepler_name": df_koi.get("kepler_name", np.nan),
    "toi":         np.nan,
    "tid":         np.nan,
})

tess_ids = pd.DataFrame({
    "source":      "TESS",
    "kepid":       np.nan,
    "kepoi_name":  np.nan,
    "kepler_name": np.nan,
    "toi":         df_tess.get("toi", np.nan),
    "tid":         df_tess.get("tid", np.nan),
})

# concatenate exactly the same order you built combo_df (KOI then TESS)
ids_combo_data = pd.concat([koi_ids, tess_ids], axis=0, ignore_index=True)

# make sure row count matches combo_df
if len(ids_combo_data) != len(combo_df):
    print(f"⚠️ Mismatch: combo_df has {len(combo_df)} rows, ids_combo_data has {len(ids_combo_data)}")
else:
    print(f"✓ ids_combo_data aligned: {len(ids_combo_data)} rows")

# === PATCH 1: safer aligner =========================================
def _align_ids(ids_df: pd.DataFrame, target_index: pd.Index) -> pd.DataFrame:
    # try exact label align first
    if ids_df.index.equals(target_index):
        return ids_df.copy()
    # try label-based reindex (fills missing with NaN but preserves order)
    if pd.Index(target_index).isin(ids_df.index).all():
        return ids_df.loc[target_index].copy()
    # final: positional (only if lengths permit)
    n = len(target_index)
    if len(ids_df) >= n:
        return ids_df.reset_index(drop=True).iloc[np.arange(n)].copy()
    raise ValueError(
        f"IDs length ({len(ids_df)}) < target length ({n}). "
        "Rebuild IDs to match combo_df rows."
    )

# === PATCH 2: robust rebuild of ids_combo_data =======================
def _needs_rebuild(ids_df: pd.DataFrame) -> bool:
    if ids_df is None:
        return True
    key_cols = ["kepid", "kepoi_name", "kepler_name", "toi", "tid"]
    if not any(c in ids_df.columns for c in key_cols):
        return True
    non_source = [c for c in key_cols if c in ids_df.columns]
    if not non_source:
        return True
    # if everything except 'source' is NaN, rebuild
    return ids_df[non_source].notna().sum().sum() == 0

if _needs_rebuild(ids_combo_data):
    # build KOI/TESS ID tables from the original cleaned tables
    koi_ids = pd.DataFrame({
        "source": "KOI",
        "kepid":       df_koi["kepid"]       if "kepid"       in df_koi.columns else np.nan,
        "kepoi_name":  df_koi["kepoi_name"]  if "kepoi_name"  in df_koi.columns else np.nan,
        "kepler_name": df_koi["kepler_name"] if "kepler_name" in df_koi.columns else np.nan,
        "toi": np.nan,
        "tid": np.nan,
    })

    tess_ids = pd.DataFrame({
        "source": "TESS",
        "kepid": np.nan, "kepoi_name": np.nan, "kepler_name": np.nan,
        "toi": df_tess["toi"] if "toi" in df_tess.columns else np.nan,
        "tid": df_tess["tid"] if "tid" in df_tess.columns else np.nan,
    })

    # concat in the SAME logical order as features (KOI then TESS), drop old indices
    ids_combo_all = pd.concat([koi_ids, tess_ids], axis=0, ignore_index=True)

    # ensure length matches combo_df; if not, trim by source counts seen in combo_df
    if len(ids_combo_all) != len(combo_df):
        if "source" in combo_df.columns:
            koi_n = int((combo_df["source"] == "KOI").sum())
            tess_n = int((combo_df["source"] == "TESS").sum())
        else:
            # fallback: assume KOI first then TESS, infer counts from originals
            koi_n = min(len(df_koi), len(combo_df))
            tess_n = len(combo_df) - koi_n

        ids_combo_all = pd.concat(
            [koi_ids.iloc[:koi_n], tess_ids.iloc[:tess_n]],
            axis=0, ignore_index=True
        )

        # if still mismatched, final pad (rare): extend with NaN rows (preserve source if available)
        if len(ids_combo_all) < len(combo_df):
            pad = pd.DataFrame(
                np.nan,
                index=np.arange(len(combo_df) - len(ids_combo_all)),
                columns=ids_combo_all.columns
            )
            if "source" in combo_df.columns:
                pad["source"] = combo_df["source"].iloc[len(ids_combo_all):].values
            ids_combo_all = pd.concat([ids_combo_all, pad], axis=0, ignore_index=True)

    # finally, align to combo_df order/index robustly
    ids_combo_data = _align_ids(ids_combo_all, combo_df.index)
else:
    # align even when we didn't rebuild so indices match X/combos
    ids_combo_data = _align_ids(ids_combo_data, combo_df.index)

# --- train with IDs, build outputs, save --------------------------------------
art_combo = train_stack_for_dataset(prefix="combo", X=X, y=y, ids=ids_combo_data)
combo_model_outputs = build_model_outputs_table_combo(
    art_combo, ids_combo_fallback=ids_combo_data, prefix="combo"
)
combo_model_outputs.to_csv("combo_model_outputs_test.csv", index=False)



⚠️ Mismatch: combo_df has 9308 rows, ids_combo_data has 9288


ValueError: IDs length (9288) < target length (9308). Rebuild IDs to match combo_df rows.

In [30]:
# make sure these exist from your earlier prep
# df_koi, df_tess  → cleaned/filtered tables used to build combo_df
# combo_df         → your unified feature/label table (index must come from the row sources)

import numpy as np
import pandas as pd

# ids from KOI side
koi_ids = pd.DataFrame(index=df_koi.index)
koi_ids["source"] = "KOI"
for c in ["kepid", "kepoi_name", "kepler_name"]:
    koi_ids[c] = df_koi[c] if c in df_koi.columns else np.nan
koi_ids["toi"] = np.nan
koi_ids["tid"] = np.nan

# ids from TESS side
tess_ids = pd.DataFrame(index=df_tess.index)
tess_ids["source"] = "TESS"
tess_ids["kepid"] = np.nan
tess_ids["kepoi_name"] = np.nan
tess_ids["kepler_name"] = np.nan
tess_ids["toi"] = df_tess["toi"] if "toi" in df_tess.columns else np.nan
tess_ids["tid"] = df_tess["tid"] if "tid" in df_tess.columns else np.nan

# stack to match the union rows BEFORE any reindexing
ids_combo_all = pd.concat([koi_ids, tess_ids], axis=0)

# now align to combo_df exactly (critical!)
ids_combo_data = ids_combo_all.loc[combo_df.index].copy()

# quick sanity
print("ids non-null counts:")
print(ids_combo_data[["source","kepid","kepoi_name","kepler_name","toi","tid"]].notna().sum())


KeyError: '[4669, 4671, 4672, 4673, 4674, 4675, 4676, 4677, 4678, 4679, 4680, 4681, 4682, 4683, 4684, 4685, 4686, 4687, 4689, 4690, 4691, 4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699, 4700, 4701, 4702, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4710, 4711, 4712, 4713, 4714, 4716, 4717, 4718, 4719, 4720, 4721, 4722, 4723, 4724, 4725, 4726, 4728, 4729, 4731, 4732, 4733, 4734, 4735, 4736, 4737, 4739, 4740, 4741, 4742, 4743, 4744, 4745, 4746, 4747, 4748, 4749, 4750, 4751, 4752, 4753, 4754, 4755, 4756, 4758, 4759, 4760, 4761, 4762, 4763, 4764, 4765, 4766, 4768, 4770, 4771, 4772, 4773, 4774, 4775, 4776, 4777, 4778, 4779, 4780, 4781, 4783, 4784, 4788, 4789, 4790, 4791, 4792, 4793, 4794, 4795, 4797, 4798, 4799, 4800, 4801, 4802, 4804, 4805, 4806, 4807, 4808, 4809, 4810, 4811, 4812, 4813, 4814, 4815, 4816, 4817, 4818, 4819, 4820, 4821, 4822, 4823, 4824, 4825, 4826, 4827, 4828, 4829, 4830, 4831, 4832, 4833, 4834, 4835, 4836, 4837, 4838, 4839, 4840, 4841, 4842, 4843, 4844, 4845, 4846, 4847, 4848